# Chapter 11: Memory in Agents + MCP
## Topic 2: Persistent Memory Store for Repeat Senders

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 1 established long-term memory conceptually and demonstrated its core mechanism (a separate store, plus retrieval-and-injection) using a simple in-memory Python dictionary. That dictionary was intentionally a stand-in — real production memory needs to survive not just across function calls within one running program, but across the program restarting entirely, which an in-memory dict cannot do. This topic builds the actual, durable version.
- The core design question this topic addresses: given this project's own measured 22.6% repeat-sender rate (Topic 1), what does a genuinely production-appropriate persistent store for that specific use case actually look like — what schema, what write pattern, what read pattern — grounded in this project's existing, established patterns rather than an abstract, generic database design exercise.
- This connects directly to a design decision already made and justified earlier in this notebook series: Chapter 9 Topic 4 argued that customer-record lookups must be exact-match, never similarity-based, because they have exactly one correct answer per identified key. A persistent sender-memory store is architecturally the *same kind* of problem — looking up "what do we know about sender X" has exactly one correct answer (whatever's actually stored for that sender), making this a lookup-by-key problem, structurally identical to `fd_master_database.csv`'s design, not a retrieval-and-ranking problem.


### 2. Internal Working — Step by Step

**Designing the persistent store, applying Chapter 9 Topic 3's design principles directly to this new data:**

1. **Choose the key.** For sender-history memory, the natural, unique key is the sender's email address — exactly analogous to `fd_master_database.csv`'s `FD_No` being the key for account records. This is a deliberate design choice: the store is keyed by *sender identity*, not by individual email or conversation, since the entire point is aggregating what's known about one sender across multiple separate interactions.
2. **Design the schema as curated, structured fields — not a raw dump of every past email** (Topic 1's storage-selectivity principle, and Chapter 9 Topic 3's structured-output principle for `validate_fd_reference`, both applied here directly): fields like `first_seen_date`, `total_email_count`, `last_topic`, `has_unresolved_issue`, `last_contact_date` — each individually meaningful and useful on its own, not requiring the calling code to re-parse a blob to extract a specific fact.
3. **The write path: update-or-create logic, run at the end of a successful `run_agent()` call** (connecting to Topic 1's write-timing design question) — if this sender has no existing record, create one; if a record already exists, update the relevant fields (increment the email count, update the last-topic and last-contact-date, and set or clear the unresolved-issue flag based on this interaction's actual outcome) rather than overwriting the entire record and losing prior fields that this specific interaction didn't touch.
4. **The read path: a fast, exact-match lookup by sender email, run at the start of a new `run_agent()` call**, exactly mirroring `validate_fd_reference`'s own lookup pattern — and, critically, designed to return a clear "no record exists" result (not an error, not a crash) for the common case of a genuinely first-time sender, connecting directly to Chapter 9 Topic 3's found/not-found design principle.
5. **The persistence mechanism itself** — this project's established pattern (Chapter 3's CSV-backed `fd_master_database.csv`, accessed via `get_fd_record()`) is a reasonable, consistent choice for this new store too, keeping the same technology and access pattern already proven elsewhere in the project rather than introducing a completely different persistence technology just for this one new use case.


### 3. How This Is Implemented in This Project

- The new store, `sender_history.csv` (or an equivalent persistent table), follows the exact same access pattern already established for `fd_master_database.csv`: a lower-level accessor function (mirroring `get_fd_record()`) that performs the actual file/database read and write, and a tool-facing function (mirroring `validate_fd_reference()`) that shapes the result for use in the agent loop — the same clean separation of concerns Chapter 9 Topic 3 argued for, now applied to a second, genuinely different kind of persistent data.
- The write step integrates into `run_agent()` at the exact point Topic 1 identified as the natural default: when `finalize_classification` (or an equivalent definitive outcome) is reached, after the classification decision itself is made — not interspersed throughout the agent loop's intermediate steps, keeping the write path simple and tied to one clear, well-defined trigger point.
- The read step becomes a natural candidate for exactly the kind of *proactive* tool Chapter 10 Topic 6 described for `check_sender_history` — called near the start of handling a new email, before the main classification logic runs, so that any relevant prior-interaction context is available to inform the rest of the agent's reasoning from the beginning, rather than being looked up reactively only if some other signal suggests it might be needed.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Concurrent writes are a genuine risk once this moves beyond a single-process demo.** If two emails from the same sender happened to be processed concurrently (a realistic possibility at real production volume, per this project's stated 8,000-12,000 emails/day target), a naive read-modify-write pattern (read the current record, update fields, write the whole record back) risks a lost update if both processes read the old record before either writes back the new one — a real concern a CSV-file-based store handles poorly compared to a proper database with atomic update operations.
- **Schema evolution is a genuine, recurring maintenance concern.** As the project's needs grow (Topic 1's discussion of what's worth storing), the sender-history schema will likely need new fields over time — this needs a deliberate migration approach (how existing stored records are handled when a new field is added) rather than assuming the schema will simply stay fixed forever, an issue `fd_master_database.csv`'s current design doesn't have to face since it represents a stable, external data domain (FD account terms) rather than an evolving internal memory schema.
- **Debugging a wrong or missing sender-history read requires checking the exact-match key used.** Since this is a lookup-by-key mechanism (Chapter 9 Topic 4's exact-match principle) rather than fuzzy matching, a sender using a slightly different email address on a follow-up (a typo, or genuinely a different address) would correctly *not* match an existing record — this is expected, correct exact-match behavior, not a bug, though it does mean the store's genuine limitation (it can't recognize the "same person" across different email addresses) should be understood and accepted rather than treated as something needing a fuzzy-matching fix, which would reintroduce exactly the cross-record confusion risk Chapter 9 Topic 4 warned against.
- **Monitoring:** tracking the sender-history store's actual hit rate (what fraction of new emails' sender lookups find an existing record) over time is the direct, ongoing validation of Topic 1's initial 22.6% measurement — confirming the store is actually being populated and found useful at the rate the initial measurement predicted, or revealing a gap (a lower-than-expected hit rate) worth investigating.
- **Cost and latency:** an exact-match lookup by key is fast and cheap, mirroring `validate_fd_reference`'s own negligible cost profile (Chapter 10 Topic 3) — this store's read and write operations should not be a meaningful latency or cost concern at this project's current or projected scale, unlike the heavier `search_knowledge_base` retrieval tool.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **CSV-backed storage (consistent with this project's existing pattern) vs a genuine database:** CSV-backed storage is simple, consistent with `fd_master_database.csv`'s already-established pattern, and entirely adequate for this project's current scale and single-process demo context — but it doesn't handle concurrent writes safely, which becomes a genuine concern at real production volume with multiple concurrent requests. This is exactly the kind of "honest migration trigger" already discussed in this project's own Chapter 6 (vector database migration) — the same evidence-based principle applies here: stay with the simple approach until a measured, real concern (like genuine concurrent-write conflicts) justifies the added complexity of a proper database.
- **Update-in-place vs append-only history logging:** updating a single record per sender in place (this topic's recommended default) keeps the store compact and the read path simple, but loses the full history of how that sender's record changed over time. An append-only log (every interaction adds a new row, and the "current state" is derived by processing the log) preserves complete history at the cost of a more complex read path — the update-in-place approach is the right default unless a specific need for full historical trend analysis justifies the added complexity.
- **What triggers marking `has_unresolved_issue` as resolved:** this could be set explicitly by an agent's own determination during a later interaction, or it could require a separate signal (like an explicit customer confirmation, or a human reviewer's judgment) — an agent incorrectly marking its own prior unresolved issue as "resolved" without genuine confirmation risks silently dropping a real, still-open problem from memory, which is a real design risk worth deliberate consideration rather than an assumed-safe default.


### 6. Alternatives and When to Use Each

- **CSV-backed persistent store (this project's current, consistent choice):** appropriate at this project's current scale and single-process context, consistent with `fd_master_database.csv`'s already-established pattern — the right starting point.
- **A proper database with atomic update support:** the right migration target once concurrent-write conflicts become a measured, real concern at production volume, mirroring exactly the same evidence-based migration principle already established for the vector database transition in Chapter 6.
- **Update-in-place schema (this topic's recommended default):** the right choice for most use cases, where "what's currently true about this sender" matters more than a full historical log of every change.
- **Append-only history log:** worth adopting specifically if trend analysis across a sender's full interaction history (not just current state) becomes a genuine, measured need.


### 7. Common Mistakes and Production Failures

- Using a naive read-modify-write pattern against a CSV file in a genuinely concurrent, multi-process production environment, risking lost updates when two interactions from the same sender are processed close together in time.
- Storing a raw dump of every past email's full content rather than curated, structured fields, reproducing the same anti-pattern Chapter 9 Topic 3 already warned against for `validate_fd_reference`'s design.
- Not having a deliberate schema-evolution plan, leading to inconsistent or missing fields in older records as the schema grows over time.
- Attempting to fuzzy-match sender identity (to handle a customer using a slightly different email address) rather than accepting exact-match's genuine, expected limitation — reintroducing exactly the cross-record confusion risk Chapter 9 Topic 4 specifically warned against for customer data.
- Allowing an agent to mark its own previously-flagged unresolved issue as resolved without any genuine confirmation signal, risking silently losing track of a real, still-open customer problem.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why should a sender-history memory store be looked up by exact-match key rather than similarity search?
  A: Looking up "what do we know about this specific sender" has exactly one correct answer — whatever's actually stored under that sender's identity — making this a lookup-by-key problem, not a relevance-ranking problem. This mirrors exactly Chapter 9 Topic 4's argument for customer-record lookups: similarity-based matching risks confusing two different senders' records, exactly the kind of risk that measured, real cross-customer similarity findings (Chapter 7 Topic 2) already demonstrated.

- Q: Why should the sender-history store contain curated, structured fields rather than a raw dump of past email content?
  A: Structured, individually-named fields let downstream code read exactly the specific fact it needs (like whether there's an unresolved issue) without re-parsing unstructured text, mirroring exactly the same design principle Chapter 9 Topic 3 established for `validate_fd_reference`'s output. A raw content dump would also grow unboundedly and raise real data-retention concerns that curated, purpose-built fields largely avoid.

**Intermediate**

- Q: What's the risk of a naive read-modify-write pattern against this store at real production volume?
  A: If two interactions from the same sender are processed concurrently — a realistic scenario at this project's stated 8,000-12,000 emails/day target — both processes could read the current record before either writes back an update, causing one update to silently overwrite and lose the other. This is a genuine concurrency risk that a simple CSV-file-based store handles poorly, unlike a proper database with atomic update operations.

- Q: When should the write step to this store actually happen during an agent's run?
  A: The natural default, following Topic 1's write-timing discussion, is at the point a definitive outcome is reached — `finalize_classification` or equivalent — rather than continuously throughout the agent loop's intermediate steps. This keeps the write path tied to one clear trigger and avoids writing partial or premature state to a persistent store based on intermediate reasoning that might not reflect the interaction's actual final outcome.

**Advanced**

- Q: Design the schema-evolution approach for this store as the project's memory needs grow over time.
  A: New fields should be added with sensible, explicit defaults for existing records that predate the new field, rather than assuming every record will have every field populated — the read path needs to handle a record that's missing a newer field gracefully (following Chapter 9 Topic 3's found/not-found and structured-defaults discipline), and any code consuming this store should be written defensively against evolving schema rather than assuming a fixed, unchanging shape indefinitely.

- Q: A teammate proposes migrating this store from CSV to a proper database preemptively, before any concurrency issue has actually been observed. How do you evaluate this proposal?
  A: This should be evaluated with the same evidence-based discipline Chapter 6 already established for the vector database migration — CSV-backed storage is adequate at this project's current scale and remains simple and consistent with the existing `fd_master_database.csv` pattern; migrating preemptively adds real complexity without a demonstrated, measured need. The right trigger for this migration is genuine evidence of concurrent-write conflicts at real production volume, not a theoretical concern addressed before it's actually been observed to matter.

**Scenario-based**

- Q: Production monitoring shows the sender-history store's lookup hit rate is meaningfully lower than the 22.6% repeat-sender rate measured in Topic 1. Walk through your investigation.
  A: First check whether the store is actually being written to correctly and consistently on every qualifying interaction — a gap between the measured repeat-sender rate (from raw email data) and the observed hit rate (from actual store lookups) most directly suggests some fraction of interactions that should be writing records aren't successfully doing so, which points at the write path, not the read path. Also check whether sender-identity matching has any subtle inconsistency (like case-sensitivity in email addresses) that could cause a genuine repeat sender's lookup to miss an existing record due to a formatting mismatch rather than the record genuinely not existing.

**Follow-up questions to expect:**

- "How would you handle a sender who explicitly asks for their stored history to be deleted?"
- "What would change about this design if senders could be identified by phone number as well as email?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This store's design is a direct, deliberate reuse of `fd_master_database.csv`'s established pattern, applied to a genuinely different data domain** — recognizing that "persistent, keyed, structured lookup store" is a reusable architectural pattern (not something to redesign from scratch for every new kind of persistent data) is a mark of genuinely transferable system design skill.
- **The read-modify-write concurrency risk is a general, well-known database and systems engineering concern**, often addressed with techniques like optimistic locking or atomic update operations at the database level — recognizing this as an instance of a broadly studied problem, rather than something unique to this specific memory-store use case, connects this topic to a much wider body of applicable systems engineering knowledge.
- **This topic's CSV-to-database migration trigger discussion directly parallels Chapter 6's own vector-database migration discussion** — both are instances of the same "start simple, migrate only with measured evidence of need" principle, now applied to a second, different kind of persistence decision within the same project.

### 10. Quick Revision Summary (for last-minute interview prep)

> A persistent sender-history store applies Chapter 9 Topic 3's tool-design principles (structured output, curated fields, exact-match lookup) and Chapter 9 Topic 4's exact-match-only philosophy (never similarity-based) to a genuinely new kind of data: what the agent knows about a specific sender across separate interactions, keyed by sender email exactly as `fd_master_database.csv` is keyed by `FD_No`. The write path updates a sender's record at the point a definitive interaction outcome is reached; the read path performs a fast, exact-match lookup at the start of a new interaction, correctly returning a clear "no record" result for genuinely first-time senders. This project's existing CSV-backed persistence pattern is a reasonable, consistent starting choice, though genuine concurrent-write conflicts at real production volume are the measured trigger — mirroring Chapter 6's vector-database migration principle exactly — for eventually migrating to a proper database with atomic update support. Schema evolution, sender-identity exact-match limitations (no fuzzy matching across different email addresses, by design), and the risk of an agent incorrectly clearing its own flagged unresolved issue without genuine confirmation are all real, ongoing design considerations this store's maintenance needs to account for.


### Module 1: A Real CSV-Backed Persistent Store, Following `fd_master_database.csv`'s Pattern

Build the actual lower-level accessor and tool-facing function pair, following the exact separation of concerns established for validate_fd_reference.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real CSV-backed persistent store, same design pattern
# as fd_master_database.csv / get_fd_record() / validate_fd_reference()
# ------------------------------------------------------------------

import csv
import os
from datetime import datetime

SENDER_HISTORY_PATH = "/home/claude/build/sender_history.csv"
FIELDNAMES = ["sender_email", "first_seen_date", "total_email_count",
              "last_topic", "has_unresolved_issue", "last_contact_date"]


def _read_all_records() -> dict:
    """Lower-level accessor -- mirrors get_fd_record()'s role: knows
    nothing about tools or agents, just reads the persistent file."""
    if not os.path.exists(SENDER_HISTORY_PATH):
        return {}
    records = {}
    with open(SENDER_HISTORY_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row["total_email_count"] = int(row["total_email_count"])
            row["has_unresolved_issue"] = row["has_unresolved_issue"] == "True"
            records[row["sender_email"]] = row
    return records


def _write_all_records(records: dict):
    """Lower-level accessor -- the actual persistence write."""
    with open(SENDER_HISTORY_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        for record in records.values():
            writer.writerow(record)


def check_sender_history(sender_email: str) -> dict:
    """TOOL-FACING function -- mirrors validate_fd_reference()'s role:
    exact-match lookup, structured output, clear found/not-found
    (here: is_repeat_sender) distinction, input echoed back."""
    records = _read_all_records()
    record = records.get(sender_email.strip().lower())
    if record is None:
        return {"sender_email": sender_email, "is_repeat_sender": False}
    return {"sender_email": sender_email, "is_repeat_sender": True, **record}


def write_sender_interaction(sender_email: str, topic: str, unresolved: bool):
    """TOOL-FACING write function -- update-or-create logic, exactly
    per the theory's design: never overwrites fields this interaction
    didn't touch."""
    key = sender_email.strip().lower()
    records = _read_all_records()
    today = datetime.now().strftime("%Y-%m-%d")

    if key not in records:
        records[key] = {
            "sender_email": key, "first_seen_date": today, "total_email_count": 1,
            "last_topic": topic, "has_unresolved_issue": unresolved, "last_contact_date": today,
        }
    else:
        existing = records[key]
        existing["total_email_count"] += 1  # UPDATE, not overwrite
        existing["last_topic"] = topic
        existing["has_unresolved_issue"] = unresolved
        existing["last_contact_date"] = today
        # first_seen_date is DELIBERATELY left untouched -- update-in-place principle

    _write_all_records(records)


# clean slate for this demo
if os.path.exists(SENDER_HISTORY_PATH):
    os.remove(SENDER_HISTORY_PATH)

print("=" * 70)
print("MODULE 1: REAL PERSISTENT STORE BUILT")
print("=" * 70)

# a genuinely first-time sender
result = check_sender_history("new.sender@email.com")
print(f"\nFirst-time sender lookup: {result}")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL PERSISTENT STORE BUILT

First-time sender lookup: {'sender_email': 'new.sender@email.com', 'is_repeat_sender': False}

Module 1 complete. Run Module 2 next.


### Module 2: Update-or-Create Logic, Tested Across Multiple Real Interactions

Write several interactions for the same sender across separate calls, and verify the update-in-place logic correctly preserves fields while updating others.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Update-or-create logic, tested across real interactions
# ------------------------------------------------------------------

sender = "shobha.chopra@email.com"

print("=" * 70)
print("MODULE 2: UPDATE-OR-CREATE, ACROSS SEPARATE INTERACTIONS")
print("=" * 70)

# Interaction 1: first contact
write_sender_interaction(sender, topic="premature withdrawal penalty", unresolved=True)
result_1 = check_sender_history(sender)
print(f"\n[After Interaction 1]")
print(f"  {result_1}")

# Interaction 2: SEPARATE call, same sender, follow-up
write_sender_interaction(sender, topic="premature withdrawal penalty", unresolved=False)  # NOW resolved
result_2 = check_sender_history(sender)
print(f"\n[After Interaction 2 -- issue now resolved]")
print(f"  {result_2}")

# Interaction 3: a completely new topic from the same sender
write_sender_interaction(sender, topic="senior citizen interest rate query", unresolved=False)
result_3 = check_sender_history(sender)
print(f"\n[After Interaction 3 -- new topic]")
print(f"  {result_3}")

# REAL verification: did first_seen_date stay CONSTANT across all 3
# interactions, while total_email_count correctly INCREMENTED?
first_seen_constant = result_1["first_seen_date"] == result_2["first_seen_date"] == result_3["first_seen_date"]
count_incremented_correctly = (result_1["total_email_count"] == 1
                                 and result_2["total_email_count"] == 2
                                 and result_3["total_email_count"] == 3)

print(f"\nfirst_seen_date stayed CONSTANT across all 3 interactions: {first_seen_constant}")
print(f"total_email_count correctly incremented 1 -> 2 -> 3: {count_incremented_correctly}")
r1_unresolved = result_1["has_unresolved_issue"]
r2_unresolved = result_2["has_unresolved_issue"]
print(f"has_unresolved_issue correctly updated True -> False: "
      f"{r1_unresolved} -> {r2_unresolved}")

if first_seen_constant and count_incremented_correctly:
    print("\nCONFIRMED: update-in-place logic correctly PRESERVES fields this")
    print("interaction didn't touch (first_seen_date) while correctly UPDATING")
    print("fields that changed (count, topic, resolution status) -- exactly")
    print("the theory's design principle, verified with real file I/O.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: UPDATE-OR-CREATE, ACROSS SEPARATE INTERACTIONS

[After Interaction 1]
  {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'first_seen_date': '2026-07-09', 'total_email_count': 1, 'last_topic': 'premature withdrawal penalty', 'has_unresolved_issue': True, 'last_contact_date': '2026-07-09'}

[After Interaction 2 -- issue now resolved]
  {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'first_seen_date': '2026-07-09', 'total_email_count': 2, 'last_topic': 'premature withdrawal penalty', 'has_unresolved_issue': False, 'last_contact_date': '2026-07-09'}

[After Interaction 3 -- new topic]
  {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'first_seen_date': '2026-07-09', 'total_email_count': 3, 'last_topic': 'senior citizen interest rate query', 'has_unresolved_issue': False, 'last_contact_date': '2026-07-09'}

first_seen_date stayed CONSTANT across all 3 interactions: True
total_email_count correctly incremented 1 -> 2

### Module 3: Testing Against Real Repeat Senders From the Project's Actual Data

Run the real store against actual repeat senders identified in eval_set_2200.csv, confirming the store correctly distinguishes first-time from repeat contact using genuine project data.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Testing against REAL repeat senders from eval_set_2200.csv
# ------------------------------------------------------------------

import csv as csv_module
from collections import Counter

DATA_PATH = "/home/claude/repo/data/eval_set_2200.csv"
with open(DATA_PATH, newline="", encoding="utf-8") as f:
    reader = csv_module.DictReader(f)
    real_rows = list(reader)

sender_email_counts = Counter(r["email"] for r in real_rows)
real_repeat_senders = [email for email, count in sender_email_counts.items() if count > 1][:5]

print("=" * 70)
print("MODULE 3: REAL REPEAT SENDERS FROM eval_set_2200.csv, RUN THROUGH THE STORE")
print("=" * 70)

# clean slate again for this test
if os.path.exists(SENDER_HISTORY_PATH):
    os.remove(SENDER_HISTORY_PATH)

for real_sender in real_repeat_senders:
    real_sender_emails = [r for r in real_rows if r["email"] == real_sender]
    print(f"\nProcessing REAL sender '{real_sender}' ({len(real_sender_emails)} real emails in dataset):")

    for i, email_row in enumerate(real_sender_emails, start=1):
        lookup_before = check_sender_history(real_sender)
        is_repeat = lookup_before["is_repeat_sender"]
        subject_preview = email_row["subject"][:40]
        print(f"  Email {i}/{len(real_sender_emails)} -- store says is_repeat_sender={is_repeat} "
              f"(subject: '{subject_preview}...')")
        write_sender_interaction(real_sender, topic=email_row["label"], unresolved=False)

# final verification: after processing, does the store correctly show
# is_repeat_sender=True for ALL of these genuinely-repeat real senders?
all_correctly_flagged = all(
    check_sender_history(s)["is_repeat_sender"] for s in real_repeat_senders
)
print(f"\nAll {len(real_repeat_senders)} real repeat senders correctly flagged "
      f"as repeat AFTER processing: {all_correctly_flagged}")

print("\nModule 3 complete. All Chapter 11 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 11 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Same design pattern as fd_master_database.csv: lower-level accessor
  (_read_all_records/_write_all_records) SEPARATE from tool-facing
  functions (check_sender_history/write_sender_interaction).

  Update-or-create logic PRESERVES untouched fields (first_seen_date)
  while correctly updating changed ones (count, topic, resolution) --
  demonstrated with real file I/O across multiple separate calls.

  Exact-match lookup by sender email -- same found/not-found (here:
  is_repeat_sender) design principle as validate_fd_reference.

  Tested against REAL repeat senders from eval_set_2200.csv, not just
  synthetic examples -- the store correctly tracked genuine project data.

  CSV-backed storage is the right CURRENT choice, consistent with this
  project's existing pattern -- migrate to a proper database only with
  measured evidence of concurrent-write conflicts at real volume.
""")


MODULE 3: REAL REPEAT SENDERS FROM eval_set_2200.csv, RUN THROUGH THE STORE

Processing REAL sender 'deepak.pandey@gmail.com' (2 real emails in dataset):
  Email 1/2 -- store says is_repeat_sender=False (subject: 'TDS certificate needed...')
  Email 2/2 -- store says is_repeat_sender=True (subject: 'Loan status...')

Processing REAL sender 'manoj.joshi@hotmail.com' (2 real emails in dataset):
  Email 1/2 -- store says is_repeat_sender=False (subject: 'Insurance issue...')
  Email 2/2 -- store says is_repeat_sender=True (subject: 'Multiple issues...')

Processing REAL sender 'shobha.nair@gmail.com' (3 real emails in dataset):
  Email 1/3 -- store says is_repeat_sender=False (subject: 'Please read full email...')
  Email 2/3 -- store says is_repeat_sender=True (subject: 'Complaint...')
  Email 3/3 -- store says is_repeat_sender=True (subject: 'Interest not received...')

Processing REAL sender 'deepak.mehta@yahoo.com' (2 real emails in dataset):
  Email 1/2 -- store says is_repeat_sender